# Week 2c: CLIP Embedding Pipeline — CIFAR-10 Binary (Dataset 3, N=1000)
**Project**: Quantum Kernels on CLIP/BERT Embeddings — ICLR Extension

**Dataset 3 of 3**: CIFAR-10 binary — Airplane vs Automobile (vision modality)

**Why this dataset:**
- Crosses into a completely different modality (vision vs text)
- Uses CLIP image encoder — same model family but different architecture
- Binary classification — directly comparable to SST-2
- 32x32 colour images — challenging for CLIP due to low resolution, interesting test case

**Key difference from Week 2/2b:**
- CLIP outputs 512-dim embeddings (not 768-dim like BERT)
- Input is images not text — CLIPProcessor handles resizing and normalisation
- Labels remapped: airplane=0, automobile=1

**Runtime:** ~8 min on Colab T4 (CLIP is faster than BERT on these tiny images)

In [ ]:
!pip install datasets transformers torch torchvision Pillow scikit-learn numpy --quiet

import numpy as np
import time, os, pickle, shutil, json
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from transformers import CLIPModel, CLIPProcessor
import torch
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

---
## Step 0: Storage + checkpoint helpers

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/qml_project'
else:
    SAVE_DIR = 'qml_project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

def _persist_and_download(path):
    if os.path.abspath(SAVE_DIR) != os.getcwd():
        shutil.copy(path, os.path.join(SAVE_DIR, os.path.basename(path)))
        print(f"  saved -> {SAVE_DIR}/{os.path.basename(path)}")
    try:
        from google.colab import files
        files.download(path)
        print(f"  download -> {os.path.basename(path)}")
    except Exception as e:
        print(f"  (download skipped: {type(e).__name__} — saved above)")

def checkpoint(filename, save_fn):
    save_fn(filename); _persist_and_download(filename)

def save_figure(fig, name):
    for ext in ('png', 'pdf'):
        p = f"{name}.{ext}"
        fig.savefig(p, bbox_inches='tight', dpi=150)
        _persist_and_download(p)

def _find(fname):
    dp = os.path.join(SAVE_DIR, fname)
    return dp if os.path.exists(dp) else fname

print("Helpers ready.")

---
## Step 1: Load CIFAR-10 (airplane vs automobile)

We take classes 0 (airplane) and 1 (automobile) — visually distinct, balanced.
Labels remapped to 0/1 for binary classification.

Primary source: `uoft-cs/cifar10` on HuggingFace.
Fallback: `torchvision.datasets.CIFAR10` (always available in Colab).

In [ ]:
CIFAR_CLASSES = {0: 'airplane', 1: 'automobile'}
N_TRAIN_EXTRACT = 1100   # buffer for 1000 train
N_TEST_EXTRACT  = 250    # buffer for 200 test

train_images, train_labels_raw = [], []
test_images,  test_labels_raw  = [], []

# --- try HuggingFace first ---
loaded_from = None
try:
    from datasets import load_dataset
    print("Loading CIFAR-10 from uoft-cs/cifar10...")
    dataset = load_dataset("uoft-cs/cifar10")

    # column might be 'img' or 'image'
    img_col = 'img' if 'img' in dataset['train'].column_names else 'image'
    print(f"  Image column: '{img_col}'")

    for split, store_img, store_lbl, n in [
        ('train', train_images, train_labels_raw, N_TRAIN_EXTRACT * 5),
        ('test',  test_images,  test_labels_raw,  N_TEST_EXTRACT  * 5)]:
        for sample in dataset[split]:
            if sample['label'] in [0, 1]:
                store_img.append(sample[img_col])
                store_lbl.append(sample['label'])
            if len([l for l in store_lbl if l==0]) >= N_TRAIN_EXTRACT//2 and                len([l for l in store_lbl if l==1]) >= N_TRAIN_EXTRACT//2:
                break

    loaded_from = 'huggingface'
    print(f"  Loaded via HuggingFace")

except Exception as e:
    print(f"HuggingFace failed ({e}), falling back to torchvision...")
    import torchvision
    from PIL import Image

    def load_torchvision_cifar(train=True, n_per_class=600):
        ds = torchvision.datasets.CIFAR10('./data', train=train, download=True)
        imgs, lbls = [], []
        counts = {0: 0, 1: 0}
        for img, lbl in ds:
            if lbl in [0, 1] and counts[lbl] < n_per_class:
                imgs.append(img)
                lbls.append(lbl)
                counts[lbl] += 1
            if all(v >= n_per_class for v in counts.values()): break
        return imgs, lbls

    train_images, train_labels_raw = load_torchvision_cifar(train=True,  n_per_class=600)
    test_images,  test_labels_raw  = load_torchvision_cifar(train=False, n_per_class=130)
    loaded_from = 'torchvision'
    print(f"  Loaded via torchvision")

# Balance and cap both splits
import random
random.seed(42)
def balanced_subset(images, labels, n_per_class):
    c0 = [(im,lb) for im,lb in zip(images,labels) if lb==0][:n_per_class]
    c1 = [(im,lb) for im,lb in zip(images,labels) if lb==1][:n_per_class]
    combined = c0 + c1
    random.shuffle(combined)
    return [x[0] for x in combined], [x[1] for x in combined]

train_images, train_labels = balanced_subset(train_images, train_labels_raw,
                                              N_TRAIN_EXTRACT // 2)
test_images,  test_labels  = balanced_subset(test_images,  test_labels_raw,
                                              N_TEST_EXTRACT // 2)
train_labels = np.array(train_labels)
test_labels  = np.array(test_labels)

print(f"\nLoaded from: {loaded_from}")
print(f"Train: {len(train_images)} images | class 0: {np.sum(train_labels==0)} | class 1: {np.sum(train_labels==1)}")
print(f"Test : {len(test_images)} images  | class 0: {np.sum(test_labels==0)}  | class 1: {np.sum(test_labels==1)}")

# Visualise a few samples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(train_images[i])
    ax.set_title(CIFAR_CLASSES[train_labels[i]], fontsize=9)
    ax.axis('off')
plt.suptitle('Sample CIFAR-10 images (airplane vs automobile)')
plt.tight_layout(); plt.show()

---
## Step 2: Extract CLIP embeddings — checkpoint immediately after

CLIP ViT-B/32 outputs 512-dimensional image embeddings.
Images are resized from 32x32 to 224x224 internally by CLIPProcessor.
Embeddings are L2-normalised (CLIP convention) before saving.

In [ ]:
MODEL_NAME = "openai/clip-vit-base-patch32"
print(f"Loading {MODEL_NAME}...")
clip_processor = CLIPProcessor.from_pretrained(MODEL_NAME)
clip_model     = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
clip_model.eval()
print(f"CLIP loaded. Output dim: 512")

def extract_clip_embeddings(images, batch_size=64):
    """Extract L2-normalised image embeddings from CLIP ViT-B/32."""
    all_embeddings = []
    nb = (len(images) + batch_size - 1) // batch_size
    for i in range(0, len(images), batch_size):
        batch  = images[i:i+batch_size]
        inputs = clip_processor(images=batch, return_tensors='pt',
                                padding=True).to(DEVICE)
        with torch.no_grad():
            emb = clip_model.get_image_features(**inputs)
            emb = emb / emb.norm(p=2, dim=-1, keepdim=True)  # L2 normalise
        all_embeddings.append(emb.cpu().numpy())
        if (i // batch_size + 1) % 5 == 0:
            print(f"  batch {i//batch_size+1}/{nb}")
    return np.vstack(all_embeddings)

print("\nExtracting train embeddings...")
t0 = time.time()
train_emb = extract_clip_embeddings(train_images)
print(f"  done {time.time()-t0:.0f}s | {train_emb.shape}")

print("\nExtracting test embeddings...")
test_emb = extract_clip_embeddings(test_images)
print(f"  done | {test_emb.shape}")

print(f"\nEmbedding stats:")
print(f"  Shape : {train_emb.shape}  (512-dim, different from BERT 768-dim)")
print(f"  Norms : mean={np.mean(np.linalg.norm(train_emb,axis=1)):.4f} (should be ~1.0 — L2 normalised)")

print("\nCheckpoint 1 — raw CLIP embeddings:")
checkpoint('cifar10_clip_raw.npz',
           lambda p: np.savez(p, train_emb=train_emb, test_emb=test_emb,
                              train_labels=train_labels, test_labels=test_labels))

---
## Step 3: PCA (512 → 8) + normalize — checkpoint after

In [ ]:
N_QUBITS  = 8
CLIP_DIM  = 512   # CLIP ViT-B/32 output dimension (vs BERT's 768)
pca_dims  = [4, 8, 12, 16]
pca_objects = {n: PCA(n_components=n, random_state=42).fit(train_emb) for n in pca_dims}

print("PCA variance explained (CLIP 512-dim → reduced):")
for n in pca_dims:
    var = np.sum(pca_objects[n].explained_variance_ratio_)*100
    print(f"  {n:>2} dims -> {var:.1f}%")

pca_8 = pca_objects[N_QUBITS]
X_train_pca = pca_8.transform(train_emb)
X_test_pca  = pca_8.transform(test_emb)

scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train_pca)
X_test_scaled  = np.clip(scaler.transform(X_test_pca), 0, np.pi)

print(f"\nTrain range: [{X_train_scaled.min():.3f}, {X_train_scaled.max():.3f}]")
print(f"Test  range: [{X_test_scaled.min():.3f}, {X_test_scaled.max():.3f}]")

print("\nCheckpoint 2 — PCA + scaler:")
checkpoint('cifar10_pca_scaler.pkl',
           lambda p: pickle.dump({'pca': pca_8, 'scaler': scaler,
                                  'n_qubits': N_QUBITS, 'clip_dim': CLIP_DIM,
                                  'dataset': 'cifar10_binary'}, open(p,'wb')))

---
## Step 4: Classical SVM baselines on real 1000 / 200 split

In [ ]:
N_TRAIN_SVM = 1000
N_TEST_SVM  = 200
assert len(X_train_scaled) >= N_TRAIN_SVM, f"Need {N_TRAIN_SVM} train, got {len(X_train_scaled)}"
assert len(X_test_scaled)  >= N_TEST_SVM,  f"Need {N_TEST_SVM} test, got {len(X_test_scaled)}"

idx_tr = np.random.RandomState(42).choice(len(X_train_scaled), N_TRAIN_SVM, replace=False)
idx_te = np.random.RandomState(42).choice(len(X_test_scaled),  N_TEST_SVM,  replace=False)
X_tr, y_tr = X_train_scaled[idx_tr], train_labels[idx_tr]
X_te, y_te = X_test_scaled[idx_te],  test_labels[idx_te]

print(f"Train {X_tr.shape} | Test {X_te.shape}")
print(f"Class balance: train {np.mean(y_tr==1):.1%} auto | test {np.mean(y_te==1):.1%} auto")
print()

classifiers = {
    'Linear':     SVC(kernel='linear', C=1.0, random_state=42),
    'RBF':        SVC(kernel='rbf',    C=1.0, gamma='scale', random_state=42),
    'Polynomial': SVC(kernel='poly',   C=1.0, degree=3, coef0=1, random_state=42),
}
results = {}
print(f"{'Kernel':<14} {'Accuracy':>10} {'F1':>8} {'Time':>8}")
print("-"*44)
for name, clf in classifiers.items():
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te)
    results[name] = {
        'accuracy': float(accuracy_score(y_te, y_pred)),
        'f1':       float(f1_score(y_te, y_pred))
    }
    print(f"{name:<14} {results[name]['accuracy']:>10.4f} "
          f"{results[name]['f1']:>8.4f} {time.time()-t0:>6.1f}s")

print("\nCheckpoint 3 — baseline results:")
checkpoint('cifar10_baseline_results.pkl',
           lambda p: pickle.dump(results, open(p,'wb')))

---
## Step 5: PCA dimension sweep

In [ ]:
print(f"  {'Dims':>4}  {'Var%':>6}  {'Acc':>8}  {'F1':>8}")
print("-"*34)
for n_dim in pca_dims:
    X_tr_d = pca_objects[n_dim].transform(train_emb[idx_tr])
    X_te_d = pca_objects[n_dim].transform(test_emb[idx_te])
    sc_d   = MinMaxScaler(feature_range=(0, np.pi))
    X_tr_d = sc_d.fit_transform(X_tr_d)
    X_te_d = np.clip(sc_d.transform(X_te_d), 0, np.pi)
    clf_d  = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42).fit(X_tr_d, y_tr)
    yp = clf_d.predict(X_te_d)
    var = np.sum(pca_objects[n_dim].explained_variance_ratio_)*100
    print(f"  {n_dim:>4}  {var:>5.1f}%  "
          f"{accuracy_score(y_te,yp):>8.4f}  {f1_score(y_te,yp):>8.4f}")

---
## Step 6: Comparison figure (with all 3 datasets if available)

In [ ]:
# Single dataset figure
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
names  = list(results.keys())
accs   = [results[n]['accuracy'] for n in names]
f1s    = [results[n]['f1'] for n in names]
for ax, vals, metric in [(axes[0], accs, 'Accuracy'), (axes[1], f1s, 'F1')]:
    ax.bar(names, vals, color=['steelblue','coral','mediumseagreen'], edgecolor='white')
    ax.set_ylim(0.4, 1.0); ax.set_title(f'CIFAR-10 Binary {metric}')
    ax.set_ylabel(metric)
    for i, v in enumerate(vals):
        ax.text(i, v+0.005, f'{v:.3f}', ha='center', fontsize=11, fontweight='500')
plt.suptitle('Classical baselines — 8-dim CLIP embeddings\nCIFAR-10: Airplane vs Automobile (N=1000)')
plt.tight_layout(); plt.show()
save_figure(fig, 'cifar10_fig1_baselines')

# Cross-dataset 3-way comparison (if SST-2 and AG News results available)
sst2_bl, ag_bl = None, None
try:
    with open(_find('baseline_results.pkl'), 'rb') as f: sst2_bl = pickle.load(f)
except: pass
try:
    with open(_find('ag_news_baseline_results.pkl'), 'rb') as f: ag_bl = pickle.load(f)
except: pass

if sst2_bl and ag_bl:
    fig2, ax = plt.subplots(figsize=(12, 5))
    ks = ['Linear','RBF','Polynomial']
    x  = np.arange(len(ks)); w = 0.25
    ax.bar(x-w,   [sst2_bl[k]['accuracy'] for k in ks], w, label='SST-2 (BERT, sentiment)',
           color='#5B8DB8', edgecolor='white')
    ax.bar(x,     [ag_bl[k]['accuracy']   for k in ks], w, label='AG News (BERT, topic)',
           color='#2D9C4A', edgecolor='white')
    ax.bar(x+w,   [results[k]['accuracy'] for k in ks], w, label='CIFAR-10 (CLIP, vision)',
           color='#D95F02', edgecolor='white')
    ax.set_xticks(x); ax.set_xticklabels(ks)
    ax.set_ylim(0.4, 0.9); ax.set_ylabel('Accuracy')
    ax.set_title('Classical baseline accuracy across 3 datasets and modalities (N=1000)')
    ax.legend(); plt.tight_layout(); plt.show()
    save_figure(fig2, 'cross_dataset_3way_baselines')
    print("3-way cross-dataset figure saved.")
else:
    print("SST-2 or AG News baselines not found in Drive — skipping 3-way figure.")
    print("Re-run this cell after uploading baseline_results.pkl and ag_news_baseline_results.pkl.")

---
## Step 7: Save final embeddings — with shape verification

In [ ]:
def save_final(p):
    np.savez(p,
             X_train=X_train_scaled, X_test=X_test_scaled,
             y_train=train_labels,   y_test=test_labels,
             X_tr_subset=X_tr, X_te_subset=X_te,
             y_tr_subset=y_tr, y_te_subset=y_te)

print("Checkpoint 4 — final embeddings:")
checkpoint('cifar10_embeddings.npz', save_final)

loaded = np.load('cifar10_embeddings.npz')
print(f"\n  X_tr_subset = {loaded['X_tr_subset'].shape}  (must be (1000, 8))")
print(f"  X_te_subset = {loaded['X_te_subset'].shape}  (must be (200, 8))")
assert loaded['X_tr_subset'].shape == (1000, 8), "WRONG TRAIN SIZE"
assert loaded['X_te_subset'].shape == (200, 8),  "WRONG TEST SIZE"
print("\n  VERIFIED: genuine N=1000 / 200 split, 8-dim CLIP embeddings.")

meta = {
    'dataset': 'cifar10_binary',
    'classes': CIFAR_CLASSES,
    'n_classes': 2,
    'encoder': MODEL_NAME,
    'embedding_dim': CLIP_DIM,
    'n_train': 1000, 'n_test': 200,
    'n_qubits': N_QUBITS,
    'metric': 'f1_binary',
    'baselines': results
}
checkpoint('cifar10_meta.json', lambda p: json.dump(meta, open(p,'w'), indent=2))

print(f"\nAll CIFAR-10 files in {SAVE_DIR}:")
cf_files = [f for f in sorted(os.listdir(SAVE_DIR)) if 'cifar10' in f]
for f in cf_files:
    sz = os.path.getsize(os.path.join(SAVE_DIR, f))/1024
    print(f"  {f}  ({sz:.0f} KB)")

---
## Week 2c complete

### Saved + downloaded:
- `cifar10_clip_raw.npz` — raw 512-dim CLIP embeddings (checkpoint after extraction)
- `cifar10_pca_scaler.pkl` — PCA + scaler
- `cifar10_baseline_results.pkl` — 3 classical baselines
- `cifar10_embeddings.npz` — 1000/200 split, ready for Week 3c
- `cifar10_meta.json` — metadata
- `cifar10_fig1_baselines.png/.pdf`
- `cross_dataset_3way_baselines.png/.pdf` (if all 3 datasets available)

### Key difference from BERT datasets for Week 3c:
- Load `cifar10_embeddings.npz` instead of `embeddings_week2.npz`
- Metric is binary F1 (not macro F1 — only 2 classes)
- Baselines loaded from `cifar10_baseline_results.pkl`

### Important note on variance explained:
CLIP embeddings are already L2-normalised, so PCA variance numbers will look
different from BERT. Lower variance explained is expected — CLIP distributes
information more uniformly across dimensions. This is worth one sentence in the paper.

### Recovering from a crashed session:
```python
raw = np.load('/content/drive/MyDrive/qml_project/cifar10_clip_raw.npz')
train_emb, test_emb   = raw['train_emb'], raw['test_emb']
train_labels, test_labels = raw['train_labels'], raw['test_labels']
```

### Next: Week 3c
Quantum kernel on CIFAR-10 CLIP embeddings — same IQP circuit, same pipeline,
binary F1 metric. This completes your 3-dataset cross-modality study.